In [58]:
import sys
my_obj = ["Hello", "World", "Yeldos", "You"]
print(sys.getrefcount(my_obj))
new_obj = my_obj
print(sys.getrefcount(new_obj))
my_dict = {my_obj[0]: my_obj[1],
           my_obj[2]: my_obj[0]}
print(sys.getrefcount(my_obj))
print(sys.getrefcount(my_obj[3]))
del my_obj
print(sys.getrefcount(new_obj))






2
3
3
2
2


In [74]:
import sys

class CustomObject:
    def __init__(self, name):
        self.name = name

    def __del__(self):
        # Этот метод сработает ровно тогда, когда счетчик ссылок упадет до 0
        print(f"💀 [Память очищена]: {self.name} уничтожен!")

def leak_check():
    # 1. Создаем локальный объект внутри функции
    local_obj = CustomObject("ТестовыйОбъект")

    # 2. Проверяем счетчик ссылок внутри функции
    # Ожидаем 2: 1 от локальной переменной + 1 временный от getrefcount
    print(f"📸 Внутри функции (local_obj) refs: {sys.getrefcount(local_obj)}")

    # 3. Возвращаем объект наружу
    return local_obj


print("=== СЦЕНАРИЙ А (Результат игнорируется) ===")
leak_check()
print("-> Функция leak_check() отработала в Сценарии А.\n")


print("=== СЦЕНАРИЙ Б (Результат сохраняется в глобальную переменную) ===")
my_result = leak_check()
print(f"📸 Вне функции (my_result) refs: {sys.getrefcount(my_result)}")
print("-> Функция leak_check() отработала в Сценарии Б.")

=== СЦЕНАРИЙ А (Результат игнорируется) ===
📸 Внутри функции (local_obj) refs: 2
💀 [Память очищена]: ТестовыйОбъект уничтожен!
-> Функция leak_check() отработала в Сценарии А.

=== СЦЕНАРИЙ Б (Результат сохраняется в глобальную переменную) ===
📸 Внутри функции (local_obj) refs: 2
📸 Вне функции (my_result) refs: 2
-> Функция leak_check() отработала в Сценарии Б.


In [123]:
# task 3
import gc

class Person:
    def __init__(self, name, best_friend):
        self.name = name
        self.best_friend = best_friend

alice = Person("Alice", None)
dariga = Person("Dariga", alice)



print(f"getrefcont(alice):  {sys.getrefcount(alice)}")
print(f"getrefcont(dariga): {sys.getrefcount(dariga)}")

alice.best_friend = dariga
print()
print(f"getrefcont(alice):  {sys.getrefcount(alice)}")
print(f"getrefcont(dariga): {sys.getrefcount(dariga)}")

del alice
del dariga
print(gc.collect())


getrefcont(alice):  3
getrefcont(dariga): 2

getrefcont(alice):  3
getrefcont(dariga): 3
8


In [132]:
# task 4
import gc
import sys


class User:
    def __init__(self, fname):
        self.fname = fname

user = User("Yeldos")
user.hobbies = ["PickPonk", "Programming"]
user.metadata = {'id': 1}

print(sys.getrefcount(user))
print(gc.get_referents(user))
print(user.__dict__)



2
['Yeldos', ['PickPonk', 'Programming'], {'id': 1}, <class '__main__.User'>]
{'fname': 'Yeldos', 'hobbies': ['PickPonk', 'Programming'], 'metadata': {'id': 1}}


In [20]:
import sys
import weakref

class Person:
    def __init__(self, name, best_friend):
        self.name = name
        self.best_friend = best_friend

    def __del__(self):
        print(f"💀 [Память очищена]: {self.name} стерт!")

alice = Person("Alice", None)
bob = Person("Dariga", alice)
alice.best_friend = weakref.ref(bob) # Слабая ссылка!

print("--- Удаляем сильные ссылки из глобальной области ---")
del bob
# Счетчик сильных ссылок bob упал до 0 (ведь alice держала только слабую ссылку!).
# bob уничтожается мгновенно!

print(f"Проверяем слабую ссылку у alice: {alice.best_friend()}")
# Выведет None, потому что bob больше не существует!

print("--- Удаляем alice ---")
del alice
# Теперь и alice спокойно удаляется из памяти.

print()
print(gc.collect())

--- Удаляем сильные ссылки из глобальной области ---
💀 [Память очищена]: Dariga стерт!
Проверяем слабую ссылку у alice: None
--- Удаляем alice ---
💀 [Память очищена]: Alice стерт!

0
